# SEAS 8515 – Data Engineering for AI
## Homework 3

### Question 1: Data Ingestion and Transformation

#### Part A: Load and Convert

In [8]:
import pandas as pd

# Load CSV into DataFrame
df = pd.read_csv('air_quality.csv')
print("Shape:", df.shape)
print(df.head())

# Save as Parquet using PyArrow engine
df.to_parquet('air_quality.parquet', engine='pyarrow', index=False)
print("\nSaved air_quality.parquet")

Shape: (10035, 7)
        aqs id Date_local  latitude_x  longitude_x  Avg temp Centrigrade  \
0  01-073-0023   1/1/2016   33.553056      -86.815                   4.3   
1  01-073-0023   1/4/2016   33.553056      -86.815                   3.4   
2  01-073-0023   1/7/2016   33.553056      -86.815                   7.6   
3  01-073-0023  1/10/2016   33.553056      -86.815                   5.3   
4  01-073-0023  1/13/2016   33.553056      -86.815                   4.9   

   sunshine minutes  pm2.5_conc  
0               NaN         8.8  
1               NaN         6.6  
2               NaN         7.9  
3               NaN         1.9  
4               NaN         9.1  

Saved air_quality.parquet


#### Part B: Clean the Data

In [9]:
import pandas as pd

# Load Parquet
df = pd.read_parquet('air_quality.parquet', engine='pyarrow')

# Check missing values
print("Missing values before cleaning:")
print(df.isnull().sum())

# Drop predominantly empty columns (sunshine minutes is >99% missing)
threshold = 0.5
df = df.dropna(thresh=int(len(df) * threshold), axis=1)
print("\nColumns after dropping predominantly empty:", df.columns.tolist())

# Drop rows with remaining missing values
df = df.dropna()
print(f"\nShape after dropping NaN rows: {df.shape}")

# Standardize column names: lowercase and replace spaces with underscores
df.columns = df.columns.str.lower().str.replace(' ', '_')
print("Standardized columns:", df.columns.tolist())

# Show statistics
print("\nDataFrame statistics:")
print(df.describe())

# Save cleaned DataFrame
df.to_parquet('air_quality_cleaned.parquet', engine='pyarrow', index=False)
print("\nSaved air_quality_cleaned.parquet")

Missing values before cleaning:
aqs id                      0
Date_local                 41
latitude_x                 38
longitude_x                 4
Avg temp Centrigrade     1083
sunshine minutes        10028
pm2.5_conc                  0
dtype: int64

Columns after dropping predominantly empty: ['aqs id', 'Date_local', 'latitude_x', 'longitude_x', 'Avg temp Centrigrade', 'pm2.5_conc']

Shape after dropping NaN rows: (8898, 6)
Standardized columns: ['aqs_id', 'date_local', 'latitude_x', 'longitude_x', 'avg_temp_centrigrade', 'pm2.5_conc']

DataFrame statistics:
        latitude_x  longitude_x  avg_temp_centrigrade   pm2.5_conc
count  8898.000000  8898.000000           8898.000000  8898.000000
mean     35.816314  -114.452892             18.947617    12.011736
std       4.665878    12.461184              7.888162    11.369470
min      32.295150  -149.031655            -22.400000    -2.136364
25%      33.553056  -119.773210             13.400000     5.750000
50%      36.102244  -119.56

#### Part C: Feature Engineering and Filtering

In [10]:
import pandas as pd

# Load cleaned Parquet
df = pd.read_parquet('air_quality_cleaned.parquet', engine='pyarrow')

# Extract California subset (aqs_id first two characters == '06')
air_quality_california = df[df['aqs_id'].str[:2] == '06'].copy()
print(f"California rows: {air_quality_california.shape}")

# Convert temperature from Celsius to Fahrenheit
air_quality_california['temp_fahrenheit'] = air_quality_california['avg_temp_centrigrade'] * 9 / 5 + 32

# Show statistics
print("\nCalifornia DataFrame statistics:")
print(air_quality_california.describe())

California rows: (6309, 6)

California DataFrame statistics:
        latitude_x  longitude_x  avg_temp_centrigrade   pm2.5_conc  \
count  6309.000000  6309.000000           6309.000000  6309.000000   
mean     35.854003  -119.261804             18.770455    13.085160   
std       1.431530     1.055550              6.930138    12.502045   
min      33.447867  -121.623271             -0.300000    -2.136364   
25%      33.830620  -119.773210             13.600000     6.200000   
50%      36.785380  -119.773210             18.100000     9.800000   
75%      36.785380  -117.938450             23.900000    15.358333   
max      39.233477  -116.189700             41.700000   202.200000   

       temp_fahrenheit  
count      6309.000000  
mean         65.786819  
std          12.474248  
min          31.460000  
25%          56.480000  
50%          64.580000  
75%          75.020000  
max         107.060000  


#### Part D: Final Report

In [12]:
import pandas as pd
import plotly.express as px

# Parse date column
air_quality_california['date_local'] = pd.to_datetime(air_quality_california['date_local'])

# Filter for 2020
ca_2020 = air_quality_california[air_quality_california['date_local'].dt.year == 2020]

# Top 3 sensor locations by max PM2.5
top3 = (
    ca_2020.groupby(['aqs_id', 'latitude_x', 'longitude_x'])
    .agg(max_pm25=('pm2.5_conc', 'max'))
    .reset_index()
    .nlargest(3, 'max_pm25')
)

print("Top 3 California sensor locations (2020) by max PM2.5:")
print(top3)

# Plot on a tile map zoomed to California
fig = px.scatter_map(
    top3,
    lat='latitude_x',
    lon='longitude_x',
    size='max_pm25',
    color='max_pm25',
    text='aqs_id',
    hover_data={'aqs_id': True, 'max_pm25': True, 'latitude_x': True, 'longitude_x': True},
    title='Top 3 California Sensor Locations with Highest PM2.5 Concentrations (2020)',
    color_continuous_scale='YlOrRd',
    range_color=[0, top3['max_pm25'].max()],
    map_style='open-street-map',
    center={'lat': 37.0, 'lon': -119.5},
    zoom=4.5,
    size_max=30,
)
fig.show()

Top 3 California sensor locations (2020) by max PM2.5:
        aqs_id  latitude_x  longitude_x  max_pm25
3  06-057-0005   39.233477  -121.055608     202.2
0  06-019-0011   36.785380  -119.773210     171.8
1  06-031-0004   36.102244  -119.565650     144.3


### Question 2: The Four Vs of Big Data

Provide written responses below:

- Volume refers to the massive amounts of data being generated constantly across industries. Traditional relational databases and single-machine processing simply were not built to handle data at the scale organizations deal with today. Storage costs, retrieval times, and memory constraints all become real problems as data accumulates over years. Distributed systems like Hadoop and Spark address this by spreading the workload across clusters of machines rather than relying on one powerful server. Compression and columnar storage formats also help by reducing the physical footprint of large datasets without sacrificing too much accessibility.

- Velocity is the rate at which new data comes in and needs to be processed. Financial transactions, web clickstreams, and IoT sensors generate data continuously, and a pipeline that processes hourly batches may already be too slow for certain use cases. When decisions need to be made in real time, like fraud detection or traffic routing, delays in processing translate directly into worse outcomes. Stream processing tools like Kafka and Flink allow data to be ingested and acted on as it arrives rather than sitting in a queue. Building pipelines with latency requirements in mind from the start saves a lot of painful retrofitting later.

- Variety describes how data rarely comes in one clean, consistent format. Organizations pull from relational databases, APIs returning JSON, spreadsheets, PDFs, images, and plain text, often all for the same analytical goal. Each format requires different handling and makes integration harder. Schema-on-read approaches give teams flexibility to store data in its raw form and apply structure only when needed. Strong data cataloging practices also help so teams actually know what formats exist and where to find them.

- Veracity is about the reliability and accuracy of data. Even large, fast, well-integrated datasets are useless or harmful if the underlying records are wrong. Duplicate entries, sensor errors, missing fields, and inconsistent definitions across sources all quietly degrade quality. Automated validation checks, anomaly detection, and clear data ownership policies help catch problems before they spread. The harder part is building a culture where people actually report and fix data quality issues rather than working around them.

**Biggest Challenge:**
Veracity is the one I think causes the most damage in practice. The other three are difficult, but they are fundamentally solvable with the right technology and enough resources. Veracity is harder because bad data does not always look bad. A sensor that slowly drifts out of calibration, labels applied inconsistently by different people, or records that were joined incorrectly can all pass a basic sanity check while silently corrupting everything downstream. The consequences also tend to show up late, sometimes long after the bad data has been used to train models, generate reports, or inform decisions. At that point the fix is not just a pipeline change, it is rebuilding trust with stakeholders who made choices based on wrong information. The other Vs tend to produce visible, urgent failures that get addressed quickly. Poor veracity is slow and quiet, which makes it much harder to prioritize until serious damage has already been done.

**Most Common in Your Experience:**
I work in healthcare and the most damaging issue is veracity. The is a new data goverance team attempting to clean up data processes. Changing culture is the hardest. Creating documentation like metadata, data dictionaries, lineage, and transparency arouns data owners is starting to create a slow uptake in solving data trust issues more locally instead of not knowing who, what, where, when or why. The cleanup effort is entirely for the sake of implementing AI throughout the organisation.